# APIM - AI Agents

## Azure ML Model as MCP Server lab

![flow](images/azure-ml-models.gif)

Playground to deploy a trained ML model to an Azure ML online endpoint and expose it as an MCP server through Azure API Management for cloud-based agents in Foundry.

### Features
- **Azure ML Integration** - Deploy a pre-trained sklearn forecasting model to a managed online endpoint
- **MCP Server** - Expose the ML model as an MCP tool via APIM for agent consumption
- **Built-in LLM Logging** - Track token usage and tool calling with `llm-emit-token-metric`
- **Retry Policy** - Automatic retries on 429/503 errors for resilience
- **Managed Identity Auth** - Secure APIM-to-Azure ML communication without API keys

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0-- Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management)

In [1]:
import os, sys, json
from pathlib import Path

# Work around Azure CLI proxy/TLS inspection issues in this environment.
os.environ.setdefault("AZURE_CLI_DISABLE_CONNECTION_VERIFICATION", "1")

# Try to import shared notebook helpers; fallback to inline helpers when not present in this repo layout.
shared_dir = (Path.cwd() / "../../shared").resolve()
if shared_dir.exists():
    sys.path.insert(1, str(shared_dir))

try:
    import utils
except ModuleNotFoundError:
    import subprocess
    from types import SimpleNamespace

    def _print_color(msg: str, color: str = "37"):
        # ANSI colors work in most notebook terminals; plain print still works if unsupported.
        print(f"\033[{color}m{msg}\033[0m")

    def _run(cmd: str, success_msg: str = "", fail_msg: str = "", print_output: bool = True):
        proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        text = (proc.stdout or "").strip()
        err = (proc.stderr or "").strip()

        json_data = None
        if text:
            try:
                json_data = json.loads(text)
            except Exception:
                json_data = None

        ok = proc.returncode == 0
        if ok and success_msg:
            _print_color(success_msg, "32")
        if (not ok) and fail_msg:
            _print_color(f"{fail_msg}: {err or text}", "31")

        if print_output and text:
            print(text)
        if print_output and err and not ok:
            print(err)

        return SimpleNamespace(success=ok, text=text, json_data=json_data)

    def _create_resource_group(name: str, location: str):
        return _run(
            f"az group create --name {name} --location {location}",
            f"Resource group '{name}' ready",
            f"Failed to create resource group '{name}'",
        )

    def _get_deployment_output(output, key: str, label: str = ""):
        try:
            value = output.json_data["properties"]["outputs"][key]["value"]
            if label:
                _print_color(f"{label}: {value}", "36")
            return str(value)
        except Exception:
            if label:
                _print_color(f"{label}: <missing>", "33")
            return ""

    utils = SimpleNamespace(
        run=_run,
        create_resource_group=_create_resource_group,
        get_deployment_output=_get_deployment_output,
        print_ok=lambda m: _print_color(m, "32"),
        print_info=lambda m: _print_color(m, "36"),
        print_warning=lambda m: _print_color(m, "33"),
        print_error=lambda m: _print_color(m, "31"),
    )

# Derive notebook/deployment name in a way that works in VS Code and fallback environments.
nb_path = globals().get("__vsc_ipynb_file__", str(Path.cwd() / "azure-ml-models.ipynb"))
deployment_name = os.path.basename(os.path.dirname(nb_path))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"
subscription_id = ""

def build_resource_group_portal_url(subscription_id: str, resource_group: str) -> str:
    return f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}/resourceGroups/{resource_group}/overview"

def build_deployment_portal_url(subscription_id: str, resource_group: str, deployment_name: str) -> str:
    return f"https://portal.azure.com/#@/resource/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Resources/deployments/{deployment_name}/overview"

# Add timestamp suffix for resource uniqueness (avoids soft-delete conflicts on retries)
import time
ts_suffix = str(int(time.time()) % 100000)[-5:]  # Last 5 digits of timestamp

# AI Services configuration (for the Foundry agent LLM) - add suffix for uniqueness
aiservices_config = [{"name": f"foundry1-{ts_suffix}", "location": "swedencentral"}]

# Models configuration (LLM used by the agent to call MCP tools)
models_config = [{"name": "gpt-4.1-mini", "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50}]

# APIM configuration - OPTIMIZED FOR SPEED
# Developer SKU: ~10-15 min vs Basicv2: ~30+ min for provisioning
# Consumption: ~5-8 min but has limitations
apim_sku = 'Developer'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# Inference API configuration
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
inference_api_version = "2025-03-01-preview"
foundry_project_name = deployment_name

# Azure ML configuration - use same timestamp suffix for consistency
aml_endpoint_name_prefix = f"forecast-endpoint-{ts_suffix}"
aml_model_name = f"forecast-model-{ts_suffix}"
aml_deployment_name = f"forecast-deployment-{ts_suffix}"

# OPTIMIZATION SETTINGS
FAST_MODE = True  # Set to False for standard deployment
POLLING_INTERVAL = 10 if FAST_MODE else 20  # Check status every 10s (vs 20s) in fast mode
MAX_WAIT_MINUTES = 20 if FAST_MODE else 30  # Timeout at 20 min (vs 30 min) in fast mode

utils.print_ok('Notebook initialized')
if FAST_MODE:
    utils.print_info(f"FAST MODE enabled: {POLLING_INTERVAL}s polling, {MAX_WAIT_MINUTES}m timeout")


Notebook initialized
FAST MODE enabled: 10s polling, 20m timeout


<a id='1'></a>
### 1-- Verify the Azure CLI and install the ML extension

The following commands ensure that you have the latest version of the Azure CLI and the `ml` extension for Azure Machine Learning.

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

# Install the Azure ML CLI extension only if not already present
ext_check = utils.run("az extension show --name ml", "", "", print_output=False)
if not ext_check.success:
    utils.run("az extension add --name ml -y", "Azure ML extension installed", "Failed to install Azure ML extension")
else:
    utils.print_ok("Azure ML extension already installed")

Retrieved az account
{
  "environmentName": "AzureCloud",
  "homeTenantId": "8dc52dfc-9f5a-472e-afaf-c7aed972dc40",
  "id": "90551506-5441-4adf-8ca8-6ee6cde62c95",
  "isDefault": true,
  "managedByTenants": [
    {
      "tenantId": "72f988bf-86f1-41af-91ab-2d7cd011db47"
    }
  ],
  "name": "ME-MngEnvMCAP079348-msunda-1",
  "state": "Enabled",
  "tenantDefaultDomain": "MngEnvMCAP079348.onmicrosoft.com",
  "tenantDisplayName": "Contoso",
  "tenantId": "8dc52dfc-9f5a-472e-afaf-c7aed972dc40",
  "user": {
    "name": "admin@MngEnvMCAP079348.onmicrosoft.com",
    "type": "user"
  }
}
Current user: admin@MngEnvMCAP079348.onmicrosoft.com
Tenant ID: 8dc52dfc-9f5a-472e-afaf-c7aed972dc40
Subscription ID: 90551506-5441-4adf-8ca8-6ee6cde62c95
Azure ML extension already installed


<a id='2'></a>
### 2-- Create deployment using -- Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed. This includes:
- **Azure API Management** with inference API, ML prediction API, and MCP server
- **AI Foundry** with a GPT model for the agent
- **Azure ML Workspace** with a managed online endpoint
- **Log Analytics** and **Application Insights** for monitoring

Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

In [18]:
from pathlib import Path
import time

# Define the Bicep/ARM parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "aiServicesConfig": {"value": aiservices_config},
        "modelsConfig": {"value": models_config},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "inferenceAPIPath": {"value": inference_api_path},
        "inferenceAPIType": {"value": inference_api_type},
        "foundryProjectName": {"value": foundry_project_name},
        "amlEndpointName": {"value": aml_endpoint_name_prefix},
    },
}

# Write parameters file
with open("params.json", "w") as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Select template based on available files/modules.
# In this repo, main.bicep references external ../../modules paths that may not exist,
# so fallback to main.json when those modules are missing.
workspace_dir = Path.cwd()
bicep_template = workspace_dir / "main.bicep"
modules_root = (workspace_dir / "../../modules").resolve()

if bicep_template.exists() and modules_root.exists():
    template_file = "main.bicep"
else:
    template_file = "main.json"

utils.print_info(f"Using template file: {template_file}")

# Define local fallback URL builders so this cell works even if initialization helpers were not run.
def _build_resource_group_portal_url(sub_id: str, resource_group: str) -> str:
    return f"https://portal.azure.com/#@/resource/subscriptions/{sub_id}/resourceGroups/{resource_group}/overview"

def _build_deployment_portal_url(sub_id: str, resource_group: str, dep_name: str) -> str:
    return f"https://portal.azure.com/#@/resource/subscriptions/{sub_id}/resourceGroups/{resource_group}/providers/Microsoft.Resources/deployments/{dep_name}/overview"

# Use shared helpers when available; otherwise use local fallbacks.
build_rg_url = globals().get("build_resource_group_portal_url", _build_resource_group_portal_url)
build_dep_url = globals().get("build_deployment_portal_url", _build_deployment_portal_url)

def _resolve_existing_target(base_dep_name: str, default_rg: str, default_dep: str):
    rg_exists = utils.run(f"az group exists --name {default_rg}", "", "", print_output=False)
    if rg_exists.success and rg_exists.text.strip().lower() == "true":
        return default_rg, default_dep

    rg_candidates_result = utils.run(
        f"az group list --query \"[?starts_with(name, 'lab-{base_dep_name}')].name\" -o json",
        "",
        "",
        print_output=False,
    )
    rg_candidates = rg_candidates_result.json_data if isinstance(rg_candidates_result.json_data, list) else []

    best_rg = ""
    best_dep = ""
    best_timestamp = ""

    for candidate_rg in rg_candidates:
        dep_list = utils.run(
            f"az deployment group list -g {candidate_rg} --query \"[?starts_with(name, '{base_dep_name}')].{{name:name,timestamp:properties.timestamp}}\" -o json",
            "",
            "",
            print_output=False,
        )
        deployments = dep_list.json_data if isinstance(dep_list.json_data, list) else []
        for dep in deployments:
            dep_name = str(dep.get("name", ""))
            dep_timestamp = str(dep.get("timestamp", ""))
            if dep_name and dep_timestamp >= best_timestamp:
                best_rg = str(candidate_rg)
                best_dep = dep_name
                best_timestamp = dep_timestamp

    if best_rg and best_dep:
        utils.print_warning(
            f"Default resource group '{default_rg}' was not found. Reusing deployment '{best_dep}' in resource group '{best_rg}'."
        )
        return best_rg, best_dep

    return default_rg, default_dep

base_deployment_name = os.path.basename(os.path.dirname(nb_path))
resource_group_name, deployment_name = _resolve_existing_target(
    base_deployment_name,
    resource_group_name,
    deployment_name,
    )

# Create the resource group only when we are targeting the default deployment path.
if resource_group_name == f"lab-{base_deployment_name}" and deployment_name == base_deployment_name:
    utils.create_resource_group(resource_group_name, resource_group_location)

# Ensure subscription_id is available so we can print clickable Azure Portal URLs in this cell output.
if not subscription_id:
    sub_lookup = utils.run("az account show --query id -o tsv", "", "", print_output=False)
    if sub_lookup.success and sub_lookup.text:
        subscription_id = sub_lookup.text.strip()

if subscription_id:
    resource_group_portal_url = build_rg_url(subscription_id, resource_group_name)
    deployment_status_url = build_dep_url(subscription_id, resource_group_name, deployment_name)
    utils.print_info(f"Resource Group Portal URL: {resource_group_portal_url}")
    utils.print_info(f"Deployment Status URL: {deployment_status_url}")
else:
    utils.print_warning("Subscription ID unavailable. Cannot build Azure Portal deployment URL.")

# Check current deployment state first so we can show live progress for existing runs.
show_cmd = (
    f"az deployment group show -g {resource_group_name} -n {deployment_name} "
    "--query \"{state:properties.provisioningState,timestamp:properties.timestamp,error:properties.error.message}\" -o json"
)
status = utils.run(show_cmd, "", "", print_output=False)

if status.success and status.json_data:
    current_state = str(status.json_data.get("state", "Unknown"))
    utils.print_info(f"Existing deployment state: {current_state}")

    if current_state.lower() in ["running", "accepted"]:
        utils.print_info("Deployment already in progress; polling live status every 20 seconds...")
        max_checks = 90  # ~30 minutes
        for i in range(max_checks):
            poll = utils.run(show_cmd, "", "", print_output=False)
            if not poll.success or not poll.json_data:
                utils.print_warning("Unable to read deployment status during polling. Try re-running this cell.")
                break

            state = str(poll.json_data.get("state", "Unknown"))
            stamp = str(poll.json_data.get("timestamp", ""))
            print(f"[{i+1:02d}/{max_checks}] state={state} timestamp={stamp}")

            if state.lower() == "succeeded":
                utils.print_ok("Deployment completed successfully")
                output = poll
                break
            if state.lower() in ["failed", "canceled"]:
                utils.print_error(f"Deployment ended with state={state}")
                err = poll.json_data.get("error")
                if err:
                    utils.print_error(str(err))
                output = poll
                break

            time.sleep(20)
        else:
            utils.print_warning("Deployment is still running after 30 minutes. Re-run this cell to continue polling.")
            output = poll
    elif current_state.lower() == "succeeded":
        utils.print_ok("Deployment already succeeded. Skipping new deployment run.")
        output = status
    else:
        utils.print_warning(f"Existing deployment state is {current_state}. Starting a new deployment attempt.")
        output = utils.run(
            f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file {template_file} --parameters params.json",
            f"Deployment '{deployment_name}' succeeded",
            f"Deployment '{deployment_name}' failed",
        )
else:
    # No deployment found yet; start one.
    utils.print_info("No existing deployment found. Starting a new deployment...")
    utils.create_resource_group(resource_group_name, resource_group_location)
    output = utils.run(
        f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file {template_file} --parameters params.json",
        f"Deployment '{deployment_name}' succeeded",
        f"Deployment '{deployment_name}' failed",
    )

Using template file: main.json
Resource Group Portal URL: https://portal.azure.com/#@/resource/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/resourceGroups/lab-azureml-integration-with-agents-rerun/overview
Deployment Status URL: https://portal.azure.com/#@/resource/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/resourceGroups/lab-azureml-integration-with-agents-rerun/providers/Microsoft.Resources/deployments/azureml-integration-with-agents-rerun-45902/overview
Existing deployment state: Failed
Existing deployment state is Failed. Starting a new deployment attempt.
Deployment 'azureml-integration-with-agents-rerun-45902' failed: ERROR: {"status":"Failed","error":{"code":"DeploymentFailed","target":"/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/resourceGroups/lab-azureml-integration-with-agents-rerun/providers/Microsoft.Resources/deployments/azureml-integration-with-agents-rerun-45902","message":"At least one resource deployment operation failed. Please list deployment op

<a id='3'></a>
### 3-- Get the deployment outputs

Retrieve the gateway URL, subscription keys, Azure ML workspace details, and MCP endpoint from the Bicep deployment.

In [3]:
# Obtain outputs from the main deployment when available.
# If the top-level deployment is stuck (provider polling issue), fallback to nested module deployments
# and direct resource discovery so downstream cells can still run.
output = utils.run(
    f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}",
)

# Defaults used by fallback mode
log_analytics_id = ""
apim_service_id = ""
apim_resource_gateway_url = ""
apim_subscriptions = []
api_key = ""
foundry_project_endpoint = ""
aml_workspace_name = ""
aml_endpoint_name_output = aml_endpoint_name_prefix
mcp_endpoint = ""

def _safe_get(dct, *keys):
    cur = dct
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return None
        cur = cur[k]
    return cur

# Keep a stable base name for target discovery after kernel restarts.
base_deployment_name = deployment_name

def _resolve_rerun_target():
    group_candidates = [
        resource_group_name,
        f"lab-{base_deployment_name}-rerun",
        f"lab-{base_deployment_name}",
    ]
    seen_groups = set()
    best_group = ""
    best_deployment = ""
    best_timestamp = ""

    for candidate_group in group_candidates:
        if candidate_group in seen_groups:
            continue
        seen_groups.add(candidate_group)

        group_exists = utils.run(f"az group exists --name {candidate_group}", "", "", print_output=False)
        if not (group_exists.success and group_exists.text.strip().lower() == "true"):
            continue

        deployment_list = utils.run(
            f"az deployment group list -g {candidate_group} --query \"[].{{name:name,timestamp:properties.timestamp}}\" -o json",
            "",
            "",
            print_output=False,
        )
        deployments = deployment_list.json_data if isinstance(deployment_list.json_data, list) else []
        for dep in deployments:
            dep_name = str(dep.get("name", ""))
            dep_timestamp = str(dep.get("timestamp", ""))
            if dep_name.startswith(base_deployment_name) and dep_timestamp >= best_timestamp:
                best_group = candidate_group
                best_deployment = dep_name
                best_timestamp = dep_timestamp

    return best_group, best_deployment

resolved_group, resolved_deployment = _resolve_rerun_target()
if resolved_group and resolved_deployment:
    resource_group_name = resolved_group
    deployment_name = resolved_deployment
    utils.print_info(f"Using existing deployment '{deployment_name}' in resource group '{resource_group_name}'")
else:
    utils.print_warning("Could not auto-resolve a live rerun deployment. Falling back to the current notebook-based target.")

main_outputs = _safe_get(output.json_data or {}, "properties", "outputs")

if main_outputs:
    log_analytics_id = utils.get_deployment_output(output, "logAnalyticsWorkspaceId", "Log Analytics Id")
    apim_service_id = utils.get_deployment_output(output, "apimServiceId", "APIM Service Id")
    apim_resource_gateway_url = utils.get_deployment_output(output, "apimResourceGatewayURL", "APIM API Gateway URL")

    raw_subs = utils.get_deployment_output(output, "apimSubscriptions")
    if raw_subs:
        try:
            apim_subscriptions = json.loads(raw_subs.replace("\'", "\""))
        except Exception:
            apim_subscriptions = []

    for subscription in apim_subscriptions:
        subscription_name = subscription.get("name", "")
        subscription_key = subscription.get("key", "")
        if subscription_name:
            utils.print_info(f"Subscription Name: {subscription_name}")
        if subscription_key:
            utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")

    api_key = apim_subscriptions[0].get("key", "") if apim_subscriptions else ""
    foundry_project_endpoint = utils.get_deployment_output(output, "foundryProjectEndpoint", "Foundry Project Endpoint")
    aml_workspace_name = utils.get_deployment_output(output, "amlWorkspaceName", "Azure ML Workspace")
    aml_endpoint_name_output = utils.get_deployment_output(output, "amlEndpointName", "Azure ML Endpoint") or aml_endpoint_name_prefix
    mcp_endpoint = utils.get_deployment_output(output, "mcpEndpoint", "MCP Endpoint")
else:
    utils.print_warning("Top-level deployment outputs are unavailable. Using fallback resource discovery...")

    # 1) Discover APIM service details from nested deployment outputs
    apim_mod = utils.run(
        f"az deployment group show --name apimModule -g {resource_group_name}",
        "Retrieved apimModule deployment",
        "Failed to retrieve apimModule deployment",
        print_output=False,
    )
    apim_mod_outputs = _safe_get(apim_mod.json_data or {}, "properties", "outputs") or {}

    apim_gateway = _safe_get(apim_mod_outputs, "gatewayUrl", "value")
    apim_id = _safe_get(apim_mod_outputs, "id", "value")
    apim_subs = _safe_get(apim_mod_outputs, "apimSubscriptions", "value")

    if apim_gateway:
        apim_resource_gateway_url = str(apim_gateway)
        utils.print_info(f"APIM API Gateway URL: {apim_resource_gateway_url}")
    if apim_id:
        apim_service_id = str(apim_id)
        utils.print_info(f"APIM Service Id: {apim_service_id}")
    if isinstance(apim_subs, list):
        apim_subscriptions = apim_subs
        for subscription in apim_subscriptions:
            subscription_name = subscription.get("name", "")
            subscription_key = subscription.get("key", "")
            if subscription_name:
                utils.print_info(f"Subscription Name: {subscription_name}")
            if subscription_key:
                utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
        api_key = apim_subscriptions[0].get("key", "") if apim_subscriptions else ""

    # 2) Discover Foundry project endpoint from nested deployment outputs
    foundry_mod = utils.run(
        f"az deployment group show --name foundryModule -g {resource_group_name}",
        "Retrieved foundryModule deployment",
        "Failed to retrieve foundryModule deployment",
        print_output=False,
    )
    foundry_outputs = _safe_get(foundry_mod.json_data or {}, "properties", "outputs") or {}
    ext_cfg = _safe_get(foundry_outputs, "extendedAIServicesConfig", "value")
    if isinstance(ext_cfg, list) and ext_cfg:
        foundry_project_endpoint = str(ext_cfg[0].get("foundryProjectEndpoint", ""))
        if foundry_project_endpoint:
            utils.print_info(f"Foundry Project Endpoint: {foundry_project_endpoint}")

    # 3) Discover MCP endpoint from nested deployment outputs
    mcp_mod = utils.run(
        f"az deployment group show --name mlMCPModule -g {resource_group_name}",
        "Retrieved mlMCPModule deployment",
        "Failed to retrieve mlMCPModule deployment",
        print_output=False,
    )
    mcp_outputs = _safe_get(mcp_mod.json_data or {}, "properties", "outputs") or {}
    mcp_value = _safe_get(mcp_outputs, "endpoint", "value")
    if mcp_value:
        mcp_endpoint = str(mcp_value)
        utils.print_info(f"MCP Endpoint: {mcp_endpoint}")

    # 4) Discover AML workspace and endpoint names directly from RG
    aml_ws = utils.run(
        f"az ml workspace list -g {resource_group_name} -o json",
        "Retrieved Azure ML workspaces",
        "Failed to retrieve Azure ML workspaces",
        print_output=False,
    )
    if aml_ws.success and isinstance(aml_ws.json_data, list) and aml_ws.json_data:
        aml_workspace_name = str(aml_ws.json_data[0].get("name", ""))
        if aml_workspace_name:
            utils.print_info(f"Azure ML Workspace: {aml_workspace_name}")
            endpoints = utils.run(
                f"az ml online-endpoint list -g {resource_group_name} --workspace-name {aml_workspace_name} -o json",
                "Retrieved Azure ML endpoints",
                "Failed to retrieve Azure ML endpoints",
                print_output=False,
            )
            if endpoints.success and isinstance(endpoints.json_data, list) and endpoints.json_data:
                endpoint_names = [str(item.get("name", "")) for item in endpoints.json_data if item.get("name")]
                if endpoint_names:
                    preferred = [name for name in endpoint_names if name.startswith("forecast-endpoint-")]
                    aml_endpoint_name_output = preferred[0] if preferred else endpoint_names[0]
                    utils.print_info(f"Azure ML Endpoint: {aml_endpoint_name_output}")

    # 5) Discover Log Analytics workspace id directly from RG
    law = utils.run(
        f"az monitor log-analytics workspace list -g {resource_group_name} -o json",
        "Retrieved Log Analytics workspaces",
        "Failed to retrieve Log Analytics workspaces",
        print_output=False,
    )
    if law.success and isinstance(law.json_data, list) and law.json_data:
        log_analytics_id = str(law.json_data[0].get("id", ""))
        if log_analytics_id:
            utils.print_info(f"Log Analytics Id: {log_analytics_id}")

# Basic guardrails so later cells fail fast with clear context
if not apim_resource_gateway_url:
    utils.print_warning("APIM gateway URL is empty. APIM may still be provisioning.")
if not api_key:
    utils.print_warning("APIM subscription key is empty. Check apimModule deployment outputs.")
if not aml_workspace_name:
    utils.print_warning("Azure ML workspace name is empty. Verify workspace exists in the resource group.")
if not mcp_endpoint:
    utils.print_warning("MCP endpoint is empty. Verify mlMCPModule deployment completed.")

Failed to retrieve deployment: azureml-integration-with-agents: WARNING: Connection verification disabled by environment variable AZURE_CLI_DISABLE_CONNECTION_VERIFICATION
D:\a\_work\1\s\build_scripts\windows\artifacts\cli\Lib\site-packages\urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'management.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
ERROR: (ResourceGroupNotFound) Resource group 'lab-azureml-integration-with-agents' could not be found.
Code: ResourceGroupNotFound
Message: Resource group 'lab-azureml-integration-with-agents' could not be found.
D:\a\_work\1\s\build_scripts\windows\artifacts\cli\Lib\site-packages\urllib3/connectionpool.py:1063: InsecureRequestWarning: Unverified HTTPS request is being made to host 'management.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/e

<a id='4'></a>
### 4-- Register the ML model and create a deployment

The Bicep deployment created an empty Azure ML online endpoint with AAD Token authentication and identity-based storage access. Now we register the pre-trained forecasting model and create a deployment against the endpoint.

The model in the `model/` folder is a sklearn-based time-series forecasting model trained with Azure AutoML.

In [21]:
import time

# Register the ML model and capture the version from the output
model_output = utils.run(
    f"az ml model create --name {aml_model_name} --path ./mlflow-model --type mlflow_model "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name} "
    f"--query version -o tsv",
    f"Model '{aml_model_name}' registered successfully",
    f"Failed to register model '{aml_model_name}'"
)
# The last line of output is the version (earlier lines may contain CLI warnings)
model_version = model_output.text.strip().split('\n')[-1].strip() if model_output.success else "1"
utils.print_info(f"Using model version: {model_version}")

# Reduced wait time from 5s to 2s for faster iteration in FAST_MODE
sleep_time = 2 if FAST_MODE else 5
utils.print_info(f"Waiting for {sleep_time} seconds to ensure the model is fully registered before deployment...")
time.sleep(sleep_time)

# Create deployment YAML config without requiring PyYAML
deployment_config_text = f'''$schema: https://azuremlschemas.azureedge.net/latest/managedOnlineDeployment.schema.json
name: {aml_deployment_name}
endpoint_name: {aml_endpoint_name_output}
model: azureml:{aml_model_name}:{model_version}
instance_type: Standard_DS3_v2
instance_count: 1
'''

with open('deployment.yml', 'w', encoding='utf-8') as deployment_file:
    deployment_file.write(deployment_config_text)

# Create the online deployment
utils.run(
    f"az ml online-deployment create --file deployment.yml "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Deployment '{aml_deployment_name}' created successfully",
    f"Failed to create deployment '{aml_deployment_name}'"
)

# Set 100% traffic to the deployment
utils.run(
    f"az ml online-endpoint update --name {aml_endpoint_name_output} --traffic \"{aml_deployment_name}=100\" "
    f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
    f"Traffic set to 100% for '{aml_deployment_name}'",
    f"Failed to update traffic"
)


Model 'forecast-model-33005' registered successfully
1
Using model version: 1
Waiting for 2 seconds to ensure the model is fully registered before deployment...
Deployment 'forecast-deployment-33005' created successfully
..................................................................................................................................{
  "app_insights_enabled": false,
  "creation_context": {
    "created_at": "2026-06-16T03:12:43.369656+00:00",
    "created_by": "System Administrator",
    "last_modified_at": "2026-06-16T03:12:43.369657+00:00"
  },
  "egress_public_network_access": "enabled",
  "endpoint_name": "forecast-endpoint-33005",
  "environment": "azureml:/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/resourceGroups/lab-azureml-integration-with-agents-rerun/providers/Microsoft.MachineLearningServices/workspaces/aml-bsf46e7zldirk/environments/MlflowNCDEnv-ai-ml-automl/versions/22",
  "environment_variables": {
    "AML_APP_ROOT": "/var/mlflow_resources",
    

namespace(success=True,
          text='{\n  "auth_mode": "aad_token",\n  "id": "/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/resourceGroups/lab-azureml-integration-with-agents-rerun/providers/Microsoft.MachineLearningServices/workspaces/aml-bsf46e7zldirk/onlineEndpoints/forecast-endpoint-33005",\n  "identity": {\n    "principal_id": "0fa98f03-958f-4d1a-9021-afb8d7135b70",\n    "tenant_id": "8dc52dfc-9f5a-472e-afaf-c7aed972dc40",\n    "type": "system_assigned"\n  },\n  "kind": "Managed",\n  "location": "swedencentral",\n  "mirror_traffic": {},\n  "name": "forecast-endpoint-33005",\n  "openapi_uri": "https://forecast-endpoint-33005.swedencentral.inference.ml.azure.com/swagger.json",\n  "properties": {\n    "AzureAsyncOperationUri": "https://management.azure.com/subscriptions/90551506-5441-4adf-8ca8-6ee6cde62c95/providers/Microsoft.MachineLearningServices/locations/swedencentral/mfeOperationsStatus/oeidp:0b852adf-141c-4ae9-a647-e3fa5117974b:4eb6222b-1c50-4da6-a495-78046bb7d0c4?api

<a id='5'></a>
### -- Test the ML endpoint directly via Azure CLI

Invoke the Azure ML endpoint directly to verify the model is serving predictions.

In [13]:
import json
import requests

# Prepare sample input
sample_input = {
    "input_data": {
        "index": [0],
        "columns": ["ShipToDistributorOrgRefId", "ScheduledDeliveryDate"],
        "data": [[52080158.0, "2019-12-12"]]
    }
}

# Write sample input to a temp file
with open('sample-input.json', 'w') as f:
    json.dump(sample_input, f)

def _invoke_with_cli() -> str:
    result = utils.run(
        f"az ml online-endpoint invoke --name {aml_endpoint_name_output} --request-file sample-input.json "
        f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
        "ML endpoint invoked successfully",
        "Failed to invoke ML endpoint"
    )
    if result.success:
        return result.text
    raise RuntimeError(result.text or "az ml online-endpoint invoke failed")

def _invoke_with_bearer_token() -> str:
    token_result = utils.run(
        "az account get-access-token --resource https://ml.azure.com/ --query accessToken -o tsv",
        "",
        "",
        print_output=False,
    )
    if not token_result.success or not token_result.text:
        raise RuntimeError("Failed to acquire Azure ML access token")

    access_token = token_result.text.strip()
    scoring_uri = f"https://{aml_endpoint_name_output}.swedencentral.inference.ml.azure.com/score"
    response = requests.post(
        scoring_uri,
        headers={
            "Authorization": f"Bearer {access_token}",
            "Content-Type": "application/json",
        },
        json=sample_input,
        timeout=120,
    )
    response.raise_for_status()
    return response.text

try:
    output = utils.run(
        f"az ml online-endpoint invoke --name {aml_endpoint_name_output} --request-file sample-input.json "
        f"--resource-group {resource_group_name} --workspace-name {aml_workspace_name}",
        "ML endpoint invoked successfully",
        "Failed to invoke ML endpoint"
    )
    if output.success:
        utils.print_info(f"Prediction result: {output.text}")
    else:
        raise RuntimeError(output.text or "Azure ML invoke failed")
except Exception as invoke_error:
    utils.print_warning(f"CLI invoke failed, retrying with direct scoring request: {invoke_error}")
    try:
        direct_output = _invoke_with_bearer_token()
        utils.print_info(f"Prediction result: {direct_output}")
    except Exception as direct_error:
        utils.print_warning(f"Direct scoring request could not complete in this environment: {direct_error}")
        utils.print_warning("Skipping direct endpoint verification; APIM and MCP tests can continue.")


Failed to invoke ML endpoint: WARNING: Connection verification disabled by environment variable AZURE_CLI_DISABLE_CONNECTION_VERIFICATION
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
ERROR: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)
Certificate verification failed. This typically happens when using Azure CLI behind a proxy that intercepts traffic with a self-signed certificate. Please add this certificate to the trusted CA bundle. More info: https://docs.microsoft.com/cli/azure/use-cli-effectively#work-behind-a-proxy.
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
ERROR: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1006)
Certificate verification failed. This typically happens

<a id='6'></a>
### -- Test the ML prediction API through APIM

Call the ML prediction API through the APIM gateway.

In [14]:
import requests

# Prepare the request payload (matches the OpenAPI spec)
payload = {
    "input_data": {
        "index": [0],
        "columns": ["ShipToDistributorOrgRefId", "ScheduledDeliveryDate"],
        "data": [[52080158.0, "2019-12-12"]]
    }
}

# Call through APIM
response = requests.post(
    f"{apim_resource_gateway_url}/ml-prediction/score",
    headers={"Content-Type": "application/json", "api-key": api_key},
    json=payload
)

if response.status_code == 200:
    utils.print_ok(f"ML prediction via APIM succeeded")
    utils.print_info(f"Prediction result: {response.json()}")
else:
    utils.print_error(f"Unexpected status code: {response.status_code}. Response: {response.text}")

ML prediction via APIM succeeded
Prediction result: [18.748697803628303]


<a id='7'></a>
### -- Test the MCP server connection and list tools

Connect to the MCP server and verify the `predict-forecast` tool is available.

In [15]:
import nest_asyncio
import asyncio
nest_asyncio.apply()

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def list_tools(server_url):
    async with streamablehttp_client(server_url) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            print(f"Available tools: {[tool.name for tool in tools.tools]}")
            for tool in tools.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(list_tools(mcp_endpoint))

Available tools: ['predict-forecast']
  - predict-forecast: Invoke the Azure ML managed online endpoint to generate a time-series forecasting prediction for vaccine delivery based on a distributor ID and scheduled delivery date.


<a id='8'></a>
### -- Run an OpenAI Agent with the ML prediction MCP tool

Use the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) to run an agent that calls the ML model through the MCP server.

In [5]:
import asyncio
import os

os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"  # Disable tracing to avoid non-fatal errors with Azure OpenAI

import nest_asyncio
nest_asyncio.apply()

from openai import AsyncAzureOpenAI
from agents import Agent, Runner, set_default_openai_client
from agents.mcp import MCPServerStreamableHttp
from agents.model_settings import ModelSettings


async def run_agent(server_url: str):
    client = AsyncAzureOpenAI(
        azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        api_key=api_key,
        api_version=inference_api_version
    )
    set_default_openai_client(client)

    async with MCPServerStreamableHttp(
        name="ML Prediction MCP Server",
        params={
            "url": server_url,
            "headers": {
                "agent-id": "OpenAIAgent"
            }
        },
    ) as server:
        extra_headers = {"agent-id": "OpenAIAgent"}
        agent = Agent(
            name="Forecast Assistant",
            instructions=(
                "You are an AI assistant that helps with delivery forecasting. "
                "Use the predict-forecast tool to generate predictions. "
                "The tool requires: index (array of integers, e.g. [0]), "
                "columns (always ['ShipToDistributorOrgRefId', 'ScheduledDeliveryDate']), "
                "and data (array of rows, each row is [distributor_id_as_float, 'YYYY-MM-DD']). "
                "Always call the tool and return the prediction result."
            ),
            mcp_servers=[server],
            model_settings=ModelSettings(tool_choice="required", extra_headers=extra_headers),
            model=models_config[0]['name']
        )

        message = "What is the predicted delivery quantity for distributor #52080158 order on the 12th December 2019?"
        print(f"Running: {message}")
        result = await Runner.run(starting_agent=agent, input=message)
        print(result.final_output)

if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.run_until_complete(run_agent(mcp_endpoint))


Running: What is the predicted delivery quantity for distributor #52080158 order on the 12th December 2019?
I am consistently encountering a problem with the prediction tool regarding the 'input_data' parameter. I will escalate this as a technical issue. If you have another delivery date or distributor, I can try forecasting with a different query.


<a id='9'></a>
### -- Run an Azure AI Foundry Agent with the ML prediction MCP tool

Use the [Azure AI Foundry Agent](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/model-context-protocol) to invoke the ML model through the MCP server.

In [ ]:
from azure.ai.agents.models import ListSortOrder, MessageTextContent
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
import time

McpTool = None
try:
    from azure.ai.projects.models import McpTool as _McpTool
    McpTool = _McpTool
except ImportError:
    try:
        from azure.ai.agents.models import McpTool as _McpTool
        McpTool = _McpTool
    except ImportError:
        McpTool = None

if McpTool is None:
    utils.print_warning("This Azure AI SDK version does not expose McpTool, so the Foundry agent cell is being skipped.")
else:
    project_client = AIProjectClient(
        endpoint=foundry_project_endpoint,
        credential=DefaultAzureCredential()
    )
    agents_client = project_client.agents

    # MCP tool definition pointing to the ML prediction MCP server
    mcp_tool = McpTool(
        server_label="ml_prediction",
        server_url=mcp_endpoint,
)

    prompt = "Predict the delivery quantity for distributor #52080158 on 2019-12-12 and 2019-12-13 and 2019-12-14."

    # Agent creation
    agent = agents_client.create_agent(
        model=str(models_config[0].get('name')),
        name="agent-ml-forecast",
        instructions=(
            "You are an AI agent that helps with delivery forecasting. "
            "Use the predict-forecast tool to generate predictions. "
            "The tool requires: index (array of integers, e.g. [0]), "
            "columns (always ['ShipToDistributorOrgRefId', 'ScheduledDeliveryDate']), "
            "and data (array of rows, each row is [distributor_id_as_float, 'YYYY-MM-DD']). "
            "Always call the tool and return the prediction result clearly."
        ),
        tools=mcp_tool.definitions
    )
    print(f"Created agent, agent ID: {agent.id}")
    print(f"MCP Server: {mcp_tool.server_label} at {mcp_tool.server_url}")

    # Thread creation
    thread = agents_client.threads.create()
    print(f"Created thread, thread ID: {thread.id}")

    # Message creation
    message = agents_client.messages.create(
        thread_id=thread.id,
        role="user",
        content=prompt,
    )
    print(f"Created message, message ID: {message.id}")

    mcp_tool.set_approval_mode("never")

    # Run
    run = agents_client.runs.create(thread_id=thread.id, agent_id=agent.id, tool_resources=mcp_tool.resources)
    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(2)
        run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)
        print(f"Run status: {run.status}")
    if run.status == "failed":
        print(f"Run error: {run.last_error}")

    # Get Run steps
    run_steps = agents_client.run_steps.list(thread_id=thread.id, run_id=run.id)
    print()

    for step in run_steps:
        print(f"Run step: {step.id}, status: {step.status}, type: {step.type}")
        if step.type == "tool_calls":
            print(f"Tool call details:")
            for tool_call in step.step_details.tool_calls:
                print(json.dumps(tool_call.as_dict(), indent=5))

    # Get the messages in the thread
    print("\nMessages in the thread:")
    messages = agents_client.messages.list(thread_id=thread.id, order=ListSortOrder.ASCENDING)

    for item in messages:
        last_message_content = item.content[-1]
        if isinstance(last_message_content, MessageTextContent):
            print(f"{item.role}: {last_message_content.text.value}")


This Azure AI SDK version does not expose McpTool, so the Foundry agent cell is being skipped.


: 

<a id='10'></a>
### -- Query logs to verify LLM token usage and tool calling

Query the Log Analytics workspace to verify that LLM token metrics and MCP tool calls are being logged.

Note: It may take a few minutes for logs to appear in Log Analytics.

In [ ]:
import time
import pandas as pd

utils.print_info("Waiting 60 seconds for logs to propagate...")
time.sleep(60)

# Query APIM gateway logs (all APIs including ML prediction and MCP)
query = "ApiManagementGatewayLogs " \
    "| where TimeGenerated > ago(24h) " \
    "| project TimeGenerated, OperationId, ApiId, OperationName, IsRequestSuccess, " \
    "ResponseCode, BackendMethod, BackendUrl, TotalTime, BackendTime " \
    "| order by TimeGenerated desc " \
    "| take 20"

output = utils.run(
    f'az monitor log-analytics query -w {log_analytics_id} --analytics-query "{query}"',
    "Retrieved gateway logs",
    "Failed to retrieve gateway logs"
)

if output.success and output.json_data:
    display(pd.DataFrame(output.json_data))
else:
    utils.print_warning("No gateway logs found yet. Logs may take a few more minutes to appear.")

# Query LLM token usage per API call (logged by the inference API diagnostic settings)
query = "let llmHeaderLogs = ApiManagementGatewayLlmLog " \
    "| where DeploymentName != ''; " \
    "let llmLogsWithSubscriptionId = llmHeaderLogs " \
    "| join kind=leftouter ApiManagementGatewayLogs on CorrelationId " \
    "| project " \
    "TimeGenerated, CorrelationId, SubscriptionId = ApimSubscriptionId, " \
    "DeploymentName, ModelName, PromptTokens, CompletionTokens, TotalTokens; " \
    "llmLogsWithSubscriptionId " \
    "| order by TimeGenerated desc " \
    "| take 20"

output = utils.run(
    f'az monitor log-analytics query -w {log_analytics_id} --analytics-query "{query}"',
    "Retrieved token usage logs",
    "Failed to retrieve token usage logs"
)

if output.success and output.json_data:
    display(pd.DataFrame(output.json_data))
else:
    utils.print_warning("No token usage logs found yet. Run the agent cells above first, then re-run this cell.")

<a id='clean'></a>
### --- Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.